In [1]:
import pandas as pd
import requests 
import json
import os
#import networkx as nx

import matplotlib.pyplot as plt
#from wordcloud import WordCloud

from collections import defaultdict
import spacy
#import Functions as fn
#import seaborn as sns
import numpy as np  
#from fitter import Fitter
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

In [2]:
from os import listdir
from os.path import isfile, join

In [3]:
nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
nlp = spacy.load("en_core_sci_sm") 

/opt/conda/lib/python3.11/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [5]:
pwd

'/home/eidf128/eidf128/shared/export/juliana/export/juliana'

In [6]:
home_dir = '/home/eidf128/eidf128/shared/export/juliana/export/juliana'

### Functions

##### Cleaning

In [7]:
def remove_stop_words(text):
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words("english"))
    filtered = [w for w in tokens if w.lower() not in stop_words and w.isalpha()]

    return filtered

In [8]:
def remove_special_characters(text):
    """
    This function removes special characters from the text.
    :param text: text
    :return: text without special characters
    """
    cleaned_string = re.sub(r'[?|$|.|!|@|#|%|^|&|*|(|)|-|_|+|=|;|:|,|<|>|/|{|}|[|]|~|`|\'|\"|\\]',r'', text)

    return cleaned_string

In [9]:
def clean_text(text):
    if pd.isna(text) or str(text).strip() == "":
        return ""
    text = remove_special_characters(str(text)) or ""
    
    return remove_stop_words(text)

##### Count

In [10]:
# It counts the number of words in a phrase
def count_words(phrase):
    """
    This function counts the number of words in a phrase.
    :param phrase: text

    :return: the number of words in the phrase
    """
    lenght = 0
    if len(phrase) > 1 and phrase.isspace() == True:
        lenght = 1
    
    else:
        words = phrase.split()
        lenght = len(words)
    
    return lenght

In [11]:
# It counts the number of characters in a phrase
def count_characters(phrase):
    """
    This function counts the number of characters in a phrase.
    :param phrase: text

    :return: the number of characters in the phrase
    """
    lenght = 0
    if len(phrase) < 1:
        lenght = 0
    else:
        lenght = len(phrase)
        
    return lenght

##### NLP

In [12]:
# It lemmatizes a text
def lemmatizer(phrase):
    """
    This function lemmatizes a text.
    :param phrase: text

    :return: the lemmatized text
    """
    doc = nlp(phrase)
    lemmatized_tokens = [token.lemma_ for token in doc]
    lemmatized_text = ' '.join(lemmatized_tokens)
    
    return lemmatized_text

In [13]:
# It extracts the entities from a text
def entities_recognition(phrase):
    """
    This function extracts the entities from a text.
    :param phrase: text
    :return: the entities extracted from the text
    """
    entities = []
    doc_phrase = nlp(phrase)
    entities = list(doc_phrase.ents)
    
    return entities

In [14]:
def to_text(x):
    # missing
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    # already a string
    if isinstance(x, str):
        return x
    # list/tuple/set/np array of tokens
    if isinstance(x, (list, tuple, set, np.ndarray)):
        return " ".join(map(str, x))
    # fallback: convert whatever it is to string
    return str(x)


##### Organization

In [15]:
def first_non_null(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else pd.NA

In [16]:
def join_unique_non_null(s, sep=" | "):
    s = s.dropna().astype(str)
    s = s[s.str.strip().ne("")]           # drop empty strings
    s = pd.unique(s)                      # keep unique (preserve order)
    return sep.join(s) if len(s) else pd.NA

### Reading the data and creating df

In [17]:
data_path = "/home/eidf128/eidf128/shared/export/juliana/export/juliana/idr_metadata"

In [18]:
os.chdir(data_path)

In [19]:
ll

total 0
drwxrwsr-x  4 26161  7 Mar  2 15:58 idr0000-lastname-example/
drwxrwsr-x  5 26161  4 Mar  2 15:58 idr0001-graml-sysgro/
drwxrwsr-x  3 26161  3 Mar  2 15:58 idr0002-heriche-condensation/
drwxrwsr-x  3 26161  2 Mar  2 15:58 idr0003-breker-plasticity/
drwxrwsr-x  6 26161  6 Mar  2 15:58 idr0004-thorpe-rad52/
drwxrwsr-x  4 26161  5 Mar  2 15:58 idr0005-toret-adhesion/
drwxrwsr-x  3 26161  4 Mar  2 15:58 idr0006-fong-nuclearbodies/
drwxrwsr-x  3 26161  4 Mar  2 15:58 idr0007-srikumar-sumo/
drwxrwsr-x  4 26161  7 Mar  2 15:58 idr0008-rohn-actinome/
drwxrwsr-x  4 26161  3 Mar  2 15:58 idr0009-simpson-secretion/
drwxrwsr-x  3 26161  3 Mar  2 15:58 idr0010-doil-dnadamage/
drwxrwsr-x  8 26161  9 Mar  2 15:58 idr0011-ledesmafernandez-dad4/
drwxrwsr-x  7 26161  8 Mar  2 15:59 idr0012-fuchs-cellmorph/
drwxrwsr-x  6 26161  5 Mar  2 15:59 idr0013-neumann-mitocheck/
drwxrwsr-x  4 26161 10 Mar  2 15:59 idr0015-colin-taraoceans/
drwxrwsr-x  4 26161 10 Mar  2 15:59 idr0015-UNKNOWN-taraoceans/
drw

In [20]:
subfolders = [ f.path for f in os.scandir() if f.is_dir() ]
subfolders

['./idr0092-ostrop-organoid',
 './idr0016-wawer-bioactivecompoundprofiling',
 './idr0079-hartmann-lateralline',
 './idr0067-king-yeastmeiosis',
 './idr0134-peters-bryophytes',
 './idr0051-fulton-tailbudlightsheet',
 './idr0020-barr-chtog',
 './idr0150-friedman-cafs',
 './idr0011-ledesmafernandez-dad4',
 './idr0070-kerwin-hdbr',
 './idr0068-shah-zebrafishlightsheet',
 './idr0141-sokol-skinmucosa',
 './idr0025-stadler-proteinatlas',
 './idr0146-dominguez-heartlightsheet',
 './idr0028-pascualvargas-rhogtpases',
 './idr0022-koedoot-cellmigration',
 './idr0012-fuchs-cellmorph',
 './idr0071-feldman-crisprko',
 './idr0021-lawo-pericentriolarmaterial',
 './idr0033-rohban-pathways',
 './idr0040-aymoz-singlecell',
 './idr0065-camsund-crispri',
 './idr0030-sero-yap',
 './idr0116-deboer-npod',
 './idr0099-jain-beetlelightsheet',
 './idr0107-morgan-hei10',
 './idr0036-gustafsdottir-compoundprofiling',
 './idr0032-yang-meristem',
 './idr0148-schumacher-kidneytem',
 './idr0045-reichmann-zygotespindle

In [21]:
new_list_folders = []
for element in subfolders:
    value = str(element)
    value_clean = value.replace('./','')
    new_list_folders.append(value_clean)
    value = []

new_list_folders

['idr0092-ostrop-organoid',
 'idr0016-wawer-bioactivecompoundprofiling',
 'idr0079-hartmann-lateralline',
 'idr0067-king-yeastmeiosis',
 'idr0134-peters-bryophytes',
 'idr0051-fulton-tailbudlightsheet',
 'idr0020-barr-chtog',
 'idr0150-friedman-cafs',
 'idr0011-ledesmafernandez-dad4',
 'idr0070-kerwin-hdbr',
 'idr0068-shah-zebrafishlightsheet',
 'idr0141-sokol-skinmucosa',
 'idr0025-stadler-proteinatlas',
 'idr0146-dominguez-heartlightsheet',
 'idr0028-pascualvargas-rhogtpases',
 'idr0022-koedoot-cellmigration',
 'idr0012-fuchs-cellmorph',
 'idr0071-feldman-crisprko',
 'idr0021-lawo-pericentriolarmaterial',
 'idr0033-rohban-pathways',
 'idr0040-aymoz-singlecell',
 'idr0065-camsund-crispri',
 'idr0030-sero-yap',
 'idr0116-deboer-npod',
 'idr0099-jain-beetlelightsheet',
 'idr0107-morgan-hei10',
 'idr0036-gustafsdottir-compoundprofiling',
 'idr0032-yang-meristem',
 'idr0148-schumacher-kidneytem',
 'idr0045-reichmann-zygotespindle',
 'idr0015-colin-taraoceans',
 'idr0114-lindsay-hdbr',
 'i

In [22]:
pattern = r'^idr.*\-study.txt$'
dfs_IDR = []

for folder in new_list_folders:
    os.chdir(data_path + '/'+ folder)

    onlyfiles = [f for f in listdir(data_path + '/'+ folder) if isfile(join(data_path + '/'+ folder, f))]
    onlytext = [f for f in onlyfiles if re.search(pattern, f)]

    if len(onlytext) > 1:
        print('More than one text file in ' + folder)
    #print(onlytext)

    file_name = str(onlytext)
    file_name_clean = file_name.replace('[','').replace(']','').replace('\'','')

    if len(onlytext) == 0:
        print('No text file in ' + folder)
        continue

    if file_name_clean == '[]':
        print('No text file in ' + folder)
        continue

    else:

        with open(file_name_clean, encoding='latin1') as f:
            lines = f.readlines()

            dataFrame = pd.DataFrame(lines)

            for line in lines:
                if re.findall(r'^[A-Za-z]+', line):
                    test = line.split('\t')

                    if len(test) <= 1:
                        #print(test[0] + folder)
                        header = test[0]
                        header = header.replace(' ', '_')
                        content = 'None'
                        continue

                    else:                    
                    
                        header = test[0]
                        header = header.replace(' ', '_')
                        content = test[1]

                        if content == '':
                            dataFrame[header] = 'None'

                        else:
                            dataFrame[header] = content

            dataFrame.drop(0, axis=1, inplace=True)
            dataFrame.drop_duplicates(inplace=True)

            dfs_IDR.append(dataFrame)

    os.chdir('/home/eidf128/eidf128/shared/export/juliana/export/juliana')

No text file in idr0000-lastname-example


In [25]:
type(dfs_IDR)

list

In [26]:
df_IDR = pd.concat(dfs_IDR)
df_IDR

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Key_Words,Study_Organism,Study_Organism_Term_Source_REF,Study_Organism_Term_Accession,...,CK5,Fibronectin,Location_Center_X,Location_Center_Y,SOM_nodes,pg_cluster,description,firstobjectnumber,secondobjectnumber,Phenotype_Annotation_Level
0,idr0092,A semi-automated organoid screening method dem...,compound library screen,EFO,EFO_0007553,Intestinal organoids are an excellent model to...,Organoids,Mus musculus,NCBITaxon,10090,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,idr0016,Human U2OS cells - compound cell-painting expe...,high content screen,EFO,EFO_0007550,Phenotypic profiling attempts to summarize mul...,NaN,Homo sapiens,NCBITaxon,NCBITaxon_9606,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,idr0079,An Image-Based Data-Driven Analysis of Cellula...,microscopy assay,EFO,EFO_0002909,A data-driven analysis of cell morphology and ...,image analysis,Danio rerio,NCBITaxon,7955,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,idr0067,Meiotic cellular rejuvenation is coupled to nu...,time-lapse imaging,OMIT,OMIT_0027490,Production of healthy gametes in meiosis relie...,meiosis,Saccharomyces cerevisiae\n,NCBITaxon,4932,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,idr0134,Reference BioImaging dataset to assess the phe...,histology,None,None,"Here, we present a high-quality reference data...",phenotypes,Diplophyllum albicans,NCBITaxon,264775,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,idr0023,Nuclear pore scaffold structure analyzed by su...,protein localization,EFO,GO:0008104,Much of life's essential molecular machinery c...,NaN,Homo sapiens,NCBITaxon,NCBITaxon_9606,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,idr0038,Ex vivo live cell tracking in kidney organoids...,time-lapse imaging,OMIT,OMIT_0027490,We have adapted the mouse kidney rudiment assa...,NaN,Mus musculus,NCBITaxon,NCBITaxon_10090,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,idr0138,Integration of spatial and single-cell transcr...,seqFISH,EFO,EFO_0008991,Molecular profiling of single cells has advanc...,seqFISH,Mus musculus,NCBITaxon,10090,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,idr0128,Parallel compound screen for infection modulat...,high content screen,EFO,EFO_0007550,Viral infectious diseases span a myriad of mal...,Influenza A virus,Homo sapiens,NCBITaxon,9606,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Cleaning

In [27]:
# Replace None with nothing
df_IDR.replace(to_replace = [None], value="", inplace = True)

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Key_Words,Study_Organism,Study_Organism_Term_Source_REF,Study_Organism_Term_Accession,...,CK5,Fibronectin,Location_Center_X,Location_Center_Y,SOM_nodes,pg_cluster,description,firstobjectnumber,secondobjectnumber,Phenotype_Annotation_Level
0,idr0092,A semi-automated organoid screening method dem...,compound library screen,EFO,EFO_0007553,Intestinal organoids are an excellent model to...,Organoids,Mus musculus,NCBITaxon,10090,...,,,,,,,,,,
0,idr0016,Human U2OS cells - compound cell-painting expe...,high content screen,EFO,EFO_0007550,Phenotypic profiling attempts to summarize mul...,,Homo sapiens,NCBITaxon,NCBITaxon_9606,...,,,,,,,,,,
0,idr0079,An Image-Based Data-Driven Analysis of Cellula...,microscopy assay,EFO,EFO_0002909,A data-driven analysis of cell morphology and ...,image analysis,Danio rerio,NCBITaxon,7955,...,,,,,,,,,,
0,idr0067,Meiotic cellular rejuvenation is coupled to nu...,time-lapse imaging,OMIT,OMIT_0027490,Production of healthy gametes in meiosis relie...,meiosis,Saccharomyces cerevisiae\n,NCBITaxon,4932,...,,,,,,,,,,
0,idr0134,Reference BioImaging dataset to assess the phe...,histology,None,None,"Here, we present a high-quality reference data...",phenotypes,Diplophyllum albicans,NCBITaxon,264775,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,idr0023,Nuclear pore scaffold structure analyzed by su...,protein localization,EFO,GO:0008104,Much of life's essential molecular machinery c...,,Homo sapiens,NCBITaxon,NCBITaxon_9606,...,,,,,,,,,,
0,idr0038,Ex vivo live cell tracking in kidney organoids...,time-lapse imaging,OMIT,OMIT_0027490,We have adapted the mouse kidney rudiment assa...,,Mus musculus,NCBITaxon,NCBITaxon_10090,...,,,,,,,,,,
0,idr0138,Integration of spatial and single-cell transcr...,seqFISH,EFO,EFO_0008991,Molecular profiling of single cells has advanc...,seqFISH,Mus musculus,NCBITaxon,10090,...,,,,,,,,,,
0,idr0128,Parallel compound screen for infection modulat...,high content screen,EFO,EFO_0007550,Viral infectious diseases span a myriad of mal...,Influenza A virus,Homo sapiens,NCBITaxon,9606,...,,,,,,,,,,


In [28]:
# Everything to lowercase
obj_cols = df_IDR.select_dtypes(include = ["object", "string"]).columns #Select columns that have strings on it
df_IDR[obj_cols] = df_IDR[obj_cols].astype("string").apply(lambda s: s.str.lower())

In [29]:
df_IDR = df_IDR.reset_index(drop=True)

In [30]:
df_IDR["Study_Description_clean"] = pd.Series(index = df_IDR.index, dtype="object")

In [31]:
df_IDR["Experiment_Description_clean"] = pd.Series(index = df_IDR.index, dtype="object")

In [32]:
df2 = df_IDR.reset_index(drop=True)
col = "Study_Description"

for i, row in df2.iterrows():
    print(f"Processing row {i+1}/{len(df2)}")

    value_to_clean = row[col]
    if pd.notna(value_to_clean) and value_to_clean != "":
        df2.at[i, f"{col}_clean"] = clean_text(value_to_clean)

df_IDR = df2

Processing row 1/132
Processing row 2/132
Processing row 3/132
Processing row 4/132
Processing row 5/132
Processing row 6/132
Processing row 7/132
Processing row 8/132
Processing row 9/132
Processing row 10/132
Processing row 11/132
Processing row 12/132
Processing row 13/132
Processing row 14/132
Processing row 15/132
Processing row 16/132
Processing row 17/132
Processing row 18/132
Processing row 19/132
Processing row 20/132
Processing row 21/132
Processing row 22/132
Processing row 23/132
Processing row 24/132
Processing row 25/132
Processing row 26/132
Processing row 27/132
Processing row 28/132
Processing row 29/132
Processing row 30/132
Processing row 31/132
Processing row 32/132
Processing row 33/132
Processing row 34/132
Processing row 35/132
Processing row 36/132
Processing row 37/132
Processing row 38/132
Processing row 39/132
Processing row 40/132
Processing row 41/132
Processing row 42/132
Processing row 43/132
Processing row 44/132
Processing row 45/132
Processing row 46/1

In [35]:
df3 = df_IDR.reset_index(drop=True)
col = "Experiment_Description"

for i, row in df2.iterrows():
    print(f"Processing row {i+1}/{len(df3)}")

    value_to_clean = row[col]
    if pd.notna(value_to_clean) and value_to_clean != "":
        df3.at[i, f"{col}_clean"] = clean_text(value_to_clean)

df_IDR = df3

Processing row 1/132
Processing row 2/132
Processing row 3/132
Processing row 4/132
Processing row 5/132
Processing row 6/132
Processing row 7/132
Processing row 8/132
Processing row 9/132
Processing row 10/132
Processing row 11/132
Processing row 12/132
Processing row 13/132
Processing row 14/132
Processing row 15/132
Processing row 16/132
Processing row 17/132
Processing row 18/132
Processing row 19/132
Processing row 20/132
Processing row 21/132
Processing row 22/132
Processing row 23/132
Processing row 24/132
Processing row 25/132
Processing row 26/132
Processing row 27/132
Processing row 28/132
Processing row 29/132
Processing row 30/132
Processing row 31/132
Processing row 32/132
Processing row 33/132
Processing row 34/132
Processing row 35/132
Processing row 36/132
Processing row 37/132
Processing row 38/132
Processing row 39/132
Processing row 40/132
Processing row 41/132
Processing row 42/132
Processing row 43/132
Processing row 44/132
Processing row 45/132
Processing row 46/1

### Stats

In [36]:
df_IDR

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Key_Words,Study_Organism,Study_Organism_Term_Source_REF,Study_Organism_Term_Accession,...,Location_Center_X,Location_Center_Y,SOM_nodes,pg_cluster,description,firstobjectnumber,secondobjectnumber,Phenotype_Annotation_Level,Study_Description_clean,Experiment_Description_clean
0,idr0092,a semi-automated organoid screening method dem...,compound library screen,efo,efo_0007553,intestinal organoids are an excellent model to...,organoids,mus musculus,ncbitaxon,10090,...,,,,,,,,,"[intestinal, organoids, excellent, model, stud...",NaN
1,idr0016,human u2os cells - compound cell-painting expe...,high content screen,efo,efo_0007550,phenotypic profiling attempts to summarize mul...,,homo sapiens,ncbitaxon,ncbitaxon_9606,...,,,,,,,,,"[phenotypic, profiling, attempts, summarize, m...",NaN
2,idr0079,an image-based data-driven analysis of cellula...,microscopy assay,efo,efo_0002909,a data-driven analysis of cell morphology and ...,image analysis,danio rerio,ncbitaxon,7955,...,,,,,,,,,"[analysis, cell, morphology, intracellular, or...","[confocal, imaging, fixed, samples, zebrafish,..."
3,idr0067,meiotic cellular rejuvenation is coupled to nu...,time-lapse imaging,omit,omit_0027490,production of healthy gametes in meiosis relie...,meiosis,saccharomyces cerevisiae\n,ncbitaxon,4932,...,,,,,,,,,"[production, healthy, gametes, meiosis, relies...","[fixed, cell, fluorescence, microscopy, transm..."
4,idr0134,reference bioimaging dataset to assess the phe...,histology,none,none,"here, we present a high-quality reference data...",phenotypes,diplophyllum albicans,ncbitaxon,264775,...,,,,,,,,,"[present, reference, dataset, containing, macr...","[representative, voucher, specimens, received,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,idr0023,nuclear pore scaffold structure analyzed by su...,protein localization,efo,go:0008104,much of life's essential molecular machinery c...,,homo sapiens,ncbitaxon,ncbitaxon_9606,...,,,,,,,,,"[much, lifes, essential, molecular, machinery,...","[systematic, immunolabelling, complex, monomer..."
128,idr0038,ex vivo live cell tracking in kidney organoids...,time-lapse imaging,omit,omit_0027490,we have adapted the mouse kidney rudiment assa...,,mus musculus,ncbitaxon,ncbitaxon_10090,...,,,,,,,,,"[adapted, mouse, kidney, rudiment, assay, gene...","[adapted, mouse, kidney, rudiment, assay, gene..."
129,idr0138,integration of spatial and single-cell transcr...,seqfish,efo,efo_0008991,molecular profiling of single cells has advanc...,seqfish,mus musculus,ncbitaxon,10090,...,,,,,,,,,"[molecular, profiling, single, cells, advanced...","[seqfish, study, sagittal, sections, mouse, em..."
130,idr0128,parallel compound screen for infection modulat...,high content screen,efo,efo_0007550,viral infectious diseases span a myriad of mal...,influenza a virus,homo sapiens,ncbitaxon,9606,...,,,,,,,,,"[viral, infectious, diseases, span, myriad, ma...",NaN


In [37]:
df_IDR["Study_Description_word_count_clean"] = pd.Series(index=df_IDR.index, dtype="object")

In [38]:
df_IDR["Experiment_Description_word_count_clean"] = pd.Series(index=df_IDR.index, dtype="object")

In [39]:
df_IDR["Study_Description_word_count_clean"] = (
    df_IDR["Study_Description_clean"].apply(lambda x: len(x) if isinstance(x, list) else 0)
)

In [40]:
df_IDR["Experiment_Description_word_count_clean"] = (
    df_IDR["Experiment_Description_clean"].apply(lambda x: len(x) if isinstance(x, list) else 0)
)

In [41]:
df_IDR

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Key_Words,Study_Organism,Study_Organism_Term_Source_REF,Study_Organism_Term_Accession,...,SOM_nodes,pg_cluster,description,firstobjectnumber,secondobjectnumber,Phenotype_Annotation_Level,Study_Description_clean,Experiment_Description_clean,Study_Description_word_count_clean,Experiment_Description_word_count_clean
0,idr0092,a semi-automated organoid screening method dem...,compound library screen,efo,efo_0007553,intestinal organoids are an excellent model to...,organoids,mus musculus,ncbitaxon,10090,...,,,,,,,"[intestinal, organoids, excellent, model, stud...",NaN,89,0
1,idr0016,human u2os cells - compound cell-painting expe...,high content screen,efo,efo_0007550,phenotypic profiling attempts to summarize mul...,,homo sapiens,ncbitaxon,ncbitaxon_9606,...,,,,,,,"[phenotypic, profiling, attempts, summarize, m...",NaN,182,0
2,idr0079,an image-based data-driven analysis of cellula...,microscopy assay,efo,efo_0002909,a data-driven analysis of cell morphology and ...,image analysis,danio rerio,ncbitaxon,7955,...,,,,,,,"[analysis, cell, morphology, intracellular, or...","[confocal, imaging, fixed, samples, zebrafish,...",60,20
3,idr0067,meiotic cellular rejuvenation is coupled to nu...,time-lapse imaging,omit,omit_0027490,production of healthy gametes in meiosis relie...,meiosis,saccharomyces cerevisiae\n,ncbitaxon,4932,...,,,,,,,"[production, healthy, gametes, meiosis, relies...","[fixed, cell, fluorescence, microscopy, transm...",99,29
4,idr0134,reference bioimaging dataset to assess the phe...,histology,none,none,"here, we present a high-quality reference data...",phenotypes,diplophyllum albicans,ncbitaxon,264775,...,,,,,,,"[present, reference, dataset, containing, macr...","[representative, voucher, specimens, received,...",32,508
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,idr0023,nuclear pore scaffold structure analyzed by su...,protein localization,efo,go:0008104,much of life's essential molecular machinery c...,,homo sapiens,ncbitaxon,ncbitaxon_9606,...,,,,,,,"[much, lifes, essential, molecular, machinery,...","[systematic, immunolabelling, complex, monomer...",75,11
128,idr0038,ex vivo live cell tracking in kidney organoids...,time-lapse imaging,omit,omit_0027490,we have adapted the mouse kidney rudiment assa...,,mus musculus,ncbitaxon,ncbitaxon_10090,...,,,,,,,"[adapted, mouse, kidney, rudiment, assay, gene...","[adapted, mouse, kidney, rudiment, assay, gene...",79,161
129,idr0138,integration of spatial and single-cell transcr...,seqfish,efo,efo_0008991,molecular profiling of single cells has advanc...,seqfish,mus musculus,ncbitaxon,10090,...,,,,,,,"[molecular, profiling, single, cells, advanced...","[seqfish, study, sagittal, sections, mouse, em...",100,19
130,idr0128,parallel compound screen for infection modulat...,high content screen,efo,efo_0007550,viral infectious diseases span a myriad of mal...,influenza a virus,homo sapiens,ncbitaxon,9606,...,,,,,,,"[viral, infectious, diseases, span, myriad, ma...",NaN,99,0


In [42]:
list_headers_stats_clean = ["Study_Description_clean", "Experiment_Description_clean"]

In [43]:
# Creating the empty columns for lemma text and entities 
for header in list_headers_stats_clean:
    df_IDR[f"{header}_lemmatize"] = pd.Series(index = df_IDR.index, dtype="object")
    df_IDR[f"{header}_entities"]  = pd.Series(index = df_IDR.index, dtype="object")

In [44]:
# Lemmatisation and entity recognition
for idx, row in df_IDR.iterrows():
    for header in list_headers_stats_clean: 
        
        text = df_IDR.at[idx, header]
        join_text = to_text(text)
        
        lemma_str = lemmatizer("" if pd.isna(join_text) else str(join_text))
        title_lemma = f"{header}_lemmatize"
        df_IDR.at[idx, title_lemma] = lemma_str

        entities = entities_recognition(lemma_str)
        title_entities = f"{header}_entities"
        df_IDR.at[idx, title_entities] = list(entities)

In [45]:
df_IDR

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Key_Words,Study_Organism,Study_Organism_Term_Source_REF,Study_Organism_Term_Accession,...,secondobjectnumber,Phenotype_Annotation_Level,Study_Description_clean,Experiment_Description_clean,Study_Description_word_count_clean,Experiment_Description_word_count_clean,Study_Description_clean_lemmatize,Study_Description_clean_entities,Experiment_Description_clean_lemmatize,Experiment_Description_clean_entities
0,idr0092,a semi-automated organoid screening method dem...,compound library screen,efo,efo_0007553,intestinal organoids are an excellent model to...,organoids,mus musculus,ncbitaxon,10090,...,,,"[intestinal, organoids, excellent, model, stud...",NaN,89,0,intestinal organoid excellent model study epit...,"[(intestinal), (model, study), (epithelial, bi...",,[]
1,idr0016,human u2os cells - compound cell-painting expe...,high content screen,efo,efo_0007550,phenotypic profiling attempts to summarize mul...,,homo sapiens,ncbitaxon,ncbitaxon_9606,...,,,"[phenotypic, profiling, attempts, summarize, m...",NaN,182,0,phenotypic profiling attempt summarize multipa...,"[(phenotypic, profiling), (multiparametric, an...",,[]
2,idr0079,an image-based data-driven analysis of cellula...,microscopy assay,efo,efo_0002909,a data-driven analysis of cell morphology and ...,image analysis,danio rerio,ncbitaxon,7955,...,,,"[analysis, cell, morphology, intracellular, or...","[confocal, imaging, fixed, samples, zebrafish,...",60,20,analysis cell morphology intracellular organiz...,"[(morphology), (intracellular), (organization)...",confocal imaging fixed sample zebrafish poster...,"[(confocal, imaging), (posterior, lateral, lin..."
3,idr0067,meiotic cellular rejuvenation is coupled to nu...,time-lapse imaging,omit,omit_0027490,production of healthy gametes in meiosis relie...,meiosis,saccharomyces cerevisiae\n,ncbitaxon,4932,...,,,"[production, healthy, gametes, meiosis, relies...","[fixed, cell, fluorescence, microscopy, transm...",99,29,production healthy gamete meiosis rely quality...,"[(production), (healthy), (meiosis), (quality)...",fix cell fluorescence microscopy transmission ...,"[(cell, fluorescence, microscopy), (image), (y..."
4,idr0134,reference bioimaging dataset to assess the phe...,histology,none,none,"here, we present a high-quality reference data...",phenotypes,diplophyllum albicans,ncbitaxon,264775,...,,,"[present, reference, dataset, containing, macr...","[representative, voucher, specimens, received,...",32,508,present reference dataset contain macroscopic ...,"[(dataset), (macroscopic), (phenotypic, proper...",representative voucher specimen receive herbar...,"[(specimen), (herbaria), (sample, diplophyllum..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,idr0023,nuclear pore scaffold structure analyzed by su...,protein localization,efo,go:0008104,much of life's essential molecular machinery c...,,homo sapiens,ncbitaxon,ncbitaxon_9606,...,,,"[much, lifes, essential, molecular, machinery,...","[systematic, immunolabelling, complex, monomer...",75,11,much lifes essential molecular machinery consi...,"[(lifes), (molecular, machinery), (protein, as...",systematic immunolabelling complex monomeric e...,"[(systematic, immunolabelling), (complex), (mo..."
128,idr0038,ex vivo live cell tracking in kidney organoids...,time-lapse imaging,omit,omit_0027490,we have adapted the mouse kidney rudiment assa...,,mus musculus,ncbitaxon,ncbitaxon_10090,...,,,"[adapted, mouse, kidney, rudiment, assay, gene...","[adapted, mouse, kidney, rudiment, assay, gene...",79,161,adapt mouse kidney rudiment assay generate ren...,"[(adapt), (mouse, kidney, rudiment, assay), (r...",adapt mouse kidney rudiment assay generate ren...,"[(adapt), (mouse, kidney, rudiment, assay), (r..."
129,idr0138,integration of spatial and single-cell transcr...,seqfish,efo,efo_0008991,molecular profiling of si

### Saving the dataset for future work 

In [46]:
os.getcwd()

'/home/eidf128/eidf128/shared/export/juliana/export/juliana'

In [47]:
df_IDR.to_csv("df_IDR_collapse_20260406.csv")